# Capstone: Derive Backprop from Scratch

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 12.01 |
| Time | ~100 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/12_Capstone_Projects/02_derive_backprop_from_scratch.ipynb)

---

## 🎯 Learning Objective

Implement a minimal reverse-mode automatic differentiation engine, applying the chain rule and computation graphs from Module 02 to real code.

---

## 📐 Theory

`02.04` already built a working reverse-mode autodiff engine: a scalar `Value` class with
`__add__`, `__mul__`, `.tanh()`, and a `.backward()` that topologically sorts the recorded
computation graph and applies the chain rule's local-gradient recipe at every node. `02.10` then
derived, *by hand*, the backward-pass formulas for two specific layer types (dense, self-attention)
as standalone functions with no shared graph machinery. This capstone closes the gap between
them: extend `02.04`'s engine from scalars to arrays, add the two new operations
(matrix multiplication, softmax) that dense layers and attention need, and prove that this one
*general* engine automatically reproduces both of `02.10`'s hand-derived, layer-specific formulas
— without ever writing a dense-layer or attention-layer backward function again.

### From scalars to arrays: the same recipe, now on tensors

Every op in `02.04`'s `Value` recorded its **parents** and its **local gradient**, then
`.backward()` walked the graph in reverse, multiplying by local gradients and summing at
fan-out. Nothing about that recipe (the multivariate chain rule, `02.04`) requires the data to
be a scalar — a `Tensor` class can store a NumPy array instead of a float, and the "local
gradient" for each op becomes a rule for how to route an incoming *array*-shaped gradient
backward, rather than a single number. Two new ops are all that's needed to build both `02.10`'s
layer types on top of this:

**Matrix multiplication.** For $C=AB$, the chain rule (in the Jacobian form used throughout
`02.06`/`02.10`) gives $\partial L/\partial A = (\partial L/\partial C)\,B^\top$ and
$\partial L/\partial B = A^\top(\partial L/\partial C)$ — this is exactly `02.10`'s dense-layer
backward formula ($\partial L/\partial W = (\partial L/\partial\mathbf{z})\,\mathbf{x}^\top$,
$\partial L/\partial\mathbf{x} = W^\top(\partial L/\partial\mathbf{z})$), just written in terms
of a generic matrix product rather than one specific layer.

**Softmax.** The vector-Jacobian product `02.10` invoked without deriving —
$\partial L/\partial z_j = s_j\big(\partial L/\partial s_j - \sum_k s_k\,\partial L/\partial
s_k\big)$, using the Jacobian $J=\text{diag}(\mathbf{s})-\mathbf{s}\mathbf{s}^\top$ this repo's
capstone `12.01` derived from the quotient rule — becomes just one more op's local-gradient
rule, invoked automatically by `.backward()` instead of manually by the user.

### Why this is the actual point of the exercise

Once `Tensor` supports `+`, `*`, matrix multiplication, `relu`, and `softmax`, building a dense
layer or an attention layer is just *composing* these ops — exactly the way `01.10` originally
wrote `Attention(Q,K,V) = softmax(QK^T/\sqrt{d_k})V` as a formula, not as a hand-derived
gradient. The backward pass for the *entire* composed expression falls out of `.backward()`
automatically, with no per-layer-type gradient formula ever written by hand. This is precisely
what distinguishes a real autodiff engine (what `torch.autograd` is, at scale) from `02.10`'s
approach (correct, but requiring a new hand-derived backward function for every new layer type)
— verified below by reproducing `02.10`'s exact numbers using only the generic engine.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
import networkx as nx
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Drawing the computation graph a dense layer's forward pass actually traces out (following
`02.04`'s DAG convention) makes the extension from scalar to tensor nodes concrete: every node
is now a whole array, but the graph structure — and the backward-pass logic that walks it — is
identical in shape to `02.04`'s scalar example.

In [ ]:
G = nx.DiGraph()
edges = [("x (vector)", "z1 = x@W"), ("W (matrix)", "z1 = x@W"),
         ("z1 = x@W", "z2 = z1+b"), ("b (vector)", "z2 = z1+b"),
         ("z2 = z1+b", "a = relu(z2)"), ("a = relu(z2)", "L = sum(a^2)")]
G.add_edges_from(edges)
pos = {"x (vector)": (0, 2), "W (matrix)": (0, 0), "z1 = x@W": (2, 1),
       "b (vector)": (2, -1), "z2 = z1+b": (4, 0.5), "a = relu(z2)": (6, 0.5),
       "L = sum(a^2)": (8, 0.5)}

fig, ax = plt.subplots(figsize=(13, 5.5))
nx.draw_networkx_nodes(G, pos, node_size=2200, node_color="#4C72B0", alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, arrowsize=18, node_size=2200, ax=ax)
nx.draw_networkx_labels(G, pos, font_color="white", font_size=8, ax=ax)
ax.set_title("A dense layer's forward pass is the SAME kind of DAG as 02.04's scalar\nexample -- every node is now an array, and matmul/relu are the new op types")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Forward pass builds this graph automatically as operations run (each Tensor op records")
print("its own children/local-gradient rule); backward() then walks it in reverse, exactly like")
print("02.04's scalar Value.backward() -- just with array-shaped gradients flowing along each edge.")

## 🐍 Implementation from Scratch

We extend `02.04`'s scalar `Value` engine to a `Tensor` engine (array-valued nodes, plus
`matmul` and `softmax` as new ops), then verify it reproduces `02.10`'s hand-derived dense-layer
and self-attention gradients exactly — using only the generic `.backward()` traversal, with no
layer-specific backward function written anywhere.

In [ ]:
def _unbroadcast(grad, shape):
    # Sums a broadcasted gradient back down to its pre-broadcast shape (needed for e.g.
    # a bias vector broadcast-added across a batch) -- bookkeeping, not new calculus.
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad

class Tensor:
    # 02.04's Value engine, extended from scalars to arrays, with matmul and softmax added.
    def __init__(self, data, children=(), op=''):
        self.data = np.array(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._children = set(children)
        self._op = op

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += _unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += _unbroadcast(out.grad * self.data, other.data.shape)
        out._backward = _backward
        return out

    def matmul(self, other):
        # dL/dA = dL/dC @ B^T,  dL/dB = A^T @ dL/dC  (02.10's dense-layer formula, generalized)
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            a, b, g = self.data, other.data, out.grad
            a2 = a.reshape(1, -1) if a.ndim == 1 else a  # promote vectors to row matrices...
            g2 = g.reshape(1, -1) if g.ndim == 1 else g  # ...so the same formula works for both
            self.grad += (g2 @ b.T).reshape(self.data.shape)
            other.grad += (a2.T @ g2).reshape(other.data.shape)
        out._backward = _backward
        return out

    def T(self):
        out = Tensor(self.data.T, (self,), 'T')
        def _backward():
            self.grad += out.grad.T
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def softmax(self, axis=-1):
        # 02.10's vector-Jacobian product for softmax, now just ONE op's local-gradient
        # rule -- invoked automatically by backward(), never called by hand again.
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        e = np.exp(shifted)
        s = e / e.sum(axis=axis, keepdims=True)
        out = Tensor(s, (self,), 'softmax')
        def _backward():
            dot = np.sum(s * out.grad, axis=axis, keepdims=True)
            self.grad += s * (out.grad - dot)
        out._backward = _backward
        return out

    def __pow__(self, p):
        out = Tensor(self.data ** p, (self,), f'**{p}')
        def _backward():
            self.grad += (p * self.data ** (p - 1)) * out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return self + other * Tensor(-1.0)

    def sum(self):
        out = Tensor(self.data.sum(), (self,), 'sum')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # IDENTICAL topological-sort logic to 02.04's Value.backward() -- nothing new here.
        topo, visited = [], set()
        def build(v):
            if id(v) not in visited:
                visited.add(id(v))
                for c in v._children:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

# --- Verification 1: dense layer, cross-checked against 02.10's hand-derived formulas ---
rng = np.random.default_rng(0)
x0 = rng.standard_normal(4)
W0 = rng.standard_normal((4, 3)) * 0.5
b0 = rng.standard_normal(3) * 0.1

x_t, W_t, b_t = Tensor(x0), Tensor(W0), Tensor(b0)
z_t = x_t.matmul(W_t) + b_t
a_t = z_t.relu()
loss_t = (a_t ** 2).sum()
loss_t.backward()

xt, Wt, bt = (torch.tensor(v, requires_grad=True) for v in (x0, W0, b0))
torch.sum(torch.relu(xt @ Wt + bt) ** 2).backward()

print("Dense layer -- generic Tensor.backward() vs torch.autograd (no hand-derived formula used):")
print(f"  dL/dW match: {np.allclose(W_t.grad, Wt.grad.numpy())}")
print(f"  dL/db match: {np.allclose(b_t.grad, bt.grad.numpy())}")
print(f"  dL/dx match: {np.allclose(x_t.grad, xt.grad.numpy())}")

# --- Verification 2: self-attention, cross-checked against 02.10's hand-derived formulas ---
n, d_k = 4, 8
Q0, K0, V0 = (rng.standard_normal((n, d_k)) * 0.3 for _ in range(3))
Q_t, K_t, V_t = Tensor(Q0), Tensor(K0), Tensor(V0)
scale = 1 / np.sqrt(d_k)

scores_t = Q_t.matmul(K_t.T()) * scale
A_t = scores_t.softmax(axis=-1)
O_t = A_t.matmul(V_t)
loss2_t = (O_t ** 2).sum()
loss2_t.backward()

Qt, Kt, Vt = (torch.tensor(m, requires_grad=True) for m in (Q0, K0, V0))
At = torch.softmax(Qt @ Kt.T * scale, dim=-1)
torch.sum((At @ Vt) ** 2).backward()

print("\nSelf-attention -- generic Tensor.backward() vs torch.autograd (built from softmax()")
print("and matmul() as ordinary ops, NOT via 02.10's hand-written attention_backward function):")
print(f"  dL/dQ match: {np.allclose(Q_t.grad, Qt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dK match: {np.allclose(K_t.grad, Kt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dV match: {np.allclose(V_t.grad, Vt.grad.numpy(), atol=1e-6)}")

## 🤖 Why This Matters for AI

`02.04` proved its scalar engine could train a real (if tiny) neural network via nothing but
`.backward()` and plain gradient descent. This is a from-scratch reimplementation of the
algorithmic core of PyTorch's `autograd` — the exact mechanism behind every `loss.backward()`
call in every real training loop, including whatever trained the model generating this text.
The extension from scalar to tensor nodes isn't cosmetic: it's the difference between processing
one number at a time (`02.04`'s `Value`, correct but requiring one Python object per scalar —
`02.04`'s own Challenge exercise already asks you to estimate how badly that scales to
`n_hidden=4096`) and processing whole batches of vectors through matrix operations the way
`np.ndarray`/`torch.Tensor` do it natively. Below, we train a real, batched 2-layer network on
the same XOR-style task `02.04` used, but through `Tensor.matmul`/`.relu`/`.backward()` alone —
confirming the generic engine built above isn't just a gradient-checking exercise, but a genuine,
working (if minimal) training system.

In [ ]:
rng_ai = np.random.default_rng(0)

# A real, BATCHED 2-layer network -- all four XOR examples processed as one matrix, not
# one-scalar-at-a-time the way 02.04's Neuron/MLP classes had to.
W1 = Tensor(rng_ai.normal(scale=0.5, size=(2, 8)))
b1 = Tensor(np.zeros(8))
W2 = Tensor(rng_ai.normal(scale=0.5, size=(8, 1)))
b2 = Tensor(np.zeros(1))
params = [W1, b1, W2, b2]

X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([[0.0], [1.0], [1.0], [0.0]])

losses = []
for step in range(300):
    x_t = Tensor(X_xor)
    h = (x_t.matmul(W1) + b1).relu()
    pred = h.matmul(W2) + b2
    diff = pred - Tensor(y_xor)
    loss = (diff ** 2).sum()

    for p in params:
        p.grad = np.zeros_like(p.data)
    loss.backward()  # ONE call computes gradients for W1, b1, W2, b2 -- all four parameters

    lr = 0.1
    for p in params:
        p.data -= lr * p.grad

    losses.append(loss.data)

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.6f} (trained via Tensor.backward() alone, zero torch.nn)")

x_final = Tensor(X_xor)
pred_final = (x_final.matmul(W1) + b1).relu().matmul(W2) + b2
print("\nFinal predictions vs targets:")
for xi, yi, pi in zip(X_xor, y_xor, pred_final.data):
    print(f"  input={xi}, target={yi[0]}, predicted={pi[0]:.4f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(losses, color="#4C72B0", lw=2)
ax.set_yscale('log')
ax.set_xlabel("training step"); ax.set_ylabel("total squared error (log scale)")
ax.set_title("A batched network trained with the generic Tensor.backward()\nengine -- the SAME chain rule, now on whole matrices at once")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $x=(2,-1)$, $W=\begin{pmatrix}1&2\\3&-1\end{pmatrix}$, $b=(0.5,-0.5)$, and
   $L=\sum(\text{relu}(xW+b))^2$, compute $z=xW+b$, $a=\text{relu}(z)$, $L$, and every gradient
   ($\partial L/\partial x$, $\partial L/\partial W$, $\partial L/\partial b$) by hand. Verify
   with `Tensor`, and as a structural sanity check confirm the gradient is exactly $0$ at any
   coordinate where $z<0$ (ReLU's local gradient there) even though $L$ itself is nonzero.

<details>
<summary>💡 Solution</summary>

$z = xW+b$: $z_0 = 2(1)+(-1)(3)+0.5 = -0.5$, $z_1 = 2(2)+(-1)(-1)-0.5=4.5$, so $z=(-0.5,4.5)$.
Then $a=\text{relu}(z)=(0,4.5)$ (the negative entry is clipped), and $L=0^2+4.5^2=20.25$.
Backward: $\partial L/\partial a = 2a=(0,9)$; through ReLU, $\partial L/\partial z = \partial
L/\partial a$ where $z>0$ and $0$ where $z\le0$, giving $\partial L/\partial z=(0,9)$ — the
first coordinate is exactly $0$ despite $a_1=9\ne 0$ contributing to $L$, precisely because
$z_0<0$ zeroed that branch. Then $\partial L/\partial b=\partial L/\partial z=(0,9)$,
$\partial L/\partial W = x^\top(\partial L/\partial z) = \begin{pmatrix}0&18\\0&-9\end{pmatrix}$
(outer product of $x=(2,-1)$ with $(0,9)$), and $\partial L/\partial x = (\partial L/\partial
z)W^\top = (18,-9)$. Running this through `Tensor` reproduces every one of these numbers exactly.

</details>

### 💭 UNDERSTAND
2. `matmul`'s backward pass promotes 1D vectors to row matrices (`a2 = a.reshape(1,-1) if a.ndim
   == 1 else a`) before computing gradients, then reshapes back at the end. Explain what would
   go wrong — concretely, what shape mismatch or incorrect broadcast would occur — if this
   promotion step were removed and `g2 @ b.T` were computed directly on a 1D `g` when `self` is
   a vector rather than a matrix.

<details>
<summary>💡 Solution</summary>

NumPy's `@` operator treats a 1D array specially: `vector @ matrix` and `matrix @ vector`
perform genuine matrix-vector products (contracting the one shared axis), which is exactly the
semantics `matmul`'s forward pass wants, but it also means a 1D array has no distinct "row" vs
"column" orientation for `.T` to act on (`vector.T` is a no-op — transposing a 1D array returns
the same 1D array, not a column vector). Without the reshape-to-`(1,-1)` promotion, `g2 @ b.T`
with `g` genuinely 1D and `b` a matrix would still often run without raising an error (NumPy's
broadcasting rules are permissive), but it would produce a result with the wrong number of
dimensions and, in cases where the contracted dimensions don't line up unambiguously, the wrong
numerical answer entirely — silently, since NumPy won't necessarily raise a `ValueError`, it will
just contract along whichever axis happens to match in size. The promotion step exists precisely
to force every matmul-gradient computation through unambiguous 2D matrix algebra, then reshape
back to the original (possibly 1D) shape only at the very end, once there's no more ambiguity
left to introduce.

</details>

### ✏️ DERIVE
3. Derive the backward rule for `.T()` (transpose) that the engine already implements
   (`self.grad += out.grad.T`) starting from the definition of the transpose and the chain rule,
   then verify your derivation reproduces the exact code by checking it against `torch.autograd`
   on a random rectangular matrix.

<details>
<summary>💡 Solution outline</summary>

Let $B = A^\top$, so $B_{ij} = A_{ji}$. By the multivariate chain rule, the gradient of a scalar
loss $L$ with respect to $A_{ij}$ sums the contribution through every $B_{k\ell}$ that depends on
$A_{ij}$:
$$\frac{\partial L}{\partial A_{ij}} = \sum_{k,\ell}\frac{\partial L}{\partial B_{k\ell}}\cdot
\frac{\partial B_{k\ell}}{\partial A_{ij}}.$$
Since $B_{k\ell}=A_{\ell k}$, the only nonzero derivative $\partial B_{k\ell}/\partial A_{ij}$
occurs when $(\ell,k)=(i,j)$, i.e. $(k,\ell)=(j,i)$, and that single term equals $1$. So the sum
collapses to exactly one term:
$$\frac{\partial L}{\partial A_{ij}} = \frac{\partial L}{\partial B_{ji}}.$$
In matrix notation, this says $\partial L/\partial A = (\partial L/\partial B)^\top$ — the
upstream gradient, transposed — which is exactly `self.grad += out.grad.T`. Checking against
`torch.autograd` on a random $3\times5$ matrix (`A.T`, sum of squares, `.backward()`) confirms
the engine's gradient matches `A.grad` from PyTorch exactly, for every entry.

</details>

### 🐍 IMPLEMENT
4. Add three new methods to `Tensor`: `exp()` (local gradient $e^{\text{data}}$, i.e. itself),
   `sum_axis(axis)` (a partial reduction along one axis only, unlike `.sum()`'s full reduction
   to a scalar — its backward pass broadcasts the upstream gradient back out via
   `np.broadcast_to`), and `__truediv__` (division, built from `__pow__` with `p=-1`). Using
   only these three new ops plus ones the engine already has, build an *alternative* softmax as
   `exp_z / exp_z.sum_axis(axis=-1)` and check it against the hand-optimized `.softmax()` method
   on the same test input.

<details>
<summary>✅ How to know you're right</summary>

Both the forward output and the backward gradient of the `exp`/`sum_axis`/`__truediv__`
composition should match the hand-optimized `.softmax()` method exactly (to floating-point
precision, `np.allclose` with default tolerances) on the same input — since both compute
literally the same mathematical function, the generic composition of primitive ops is not
merely *an* alternative, it is required to give an identical answer via a completely different
code path through the graph. If the forward values match but the gradients don't, the bug is
almost certainly in one of the three new backward rules (a common mistake: forgetting that
`sum_axis`'s backward needs to *broadcast* the upstream gradient back out to the original
pre-reduction shape, not merely reshape it).

</details>

### 🤖 APPLY
5. Build a tiny "tied-weight autoencoder": a single weight matrix $W$ used *twice* in the same
   computation graph — once to encode (`h = x.matmul(W)`) and once, via its transpose, to decode
   back (`recon = h.matmul(W.T())`) — with a reconstruction loss `((recon - x) ** 2).sum()`.
   Call `.backward()` once and inspect `W.grad`, then check it against `torch.autograd` on the
   same tied-weight computation.

<details>
<summary>✅ What you should observe</summary>

`W.grad` should already correctly reflect *both* uses of `W` in the graph — the engine's
topological sort and accumulation (`+=`, not `=`) at every backward step means gradient
contributions from the encode path and the decode path add together automatically, with no
special handling needed anywhere for the fact that the same `Tensor` object appears twice. This
should match `torch.autograd`'s gradient on an equivalent tied-weight computation exactly. This
is precisely the mechanism that makes weight tying (e.g. sharing an embedding matrix between a
transformer's input embedding and output projection) work correctly under real autodiff: a
parameter used in multiple places simply accumulates gradient contributions from every place it
was used, with no extra bookkeeping required from the person building the model.

</details>

### 🚀 CHALLENGE
6. Every backward rule in this engine accumulates into `.grad` with `+=`. Modify a copy of
   `Tensor` so every backward rule instead *overwrites* `.grad` with `=`, and investigate what
   breaks. Test it on two cases: (a) the tied-weight autoencoder from APPLY, where `W` is used
   twice, and (b) a plain, non-tied dense-layer computation where every tensor is used exactly
   once. Characterize precisely when the bug does and doesn't manifest, and connect this to why
   real autodiff engines (`torch.autograd` included) always accumulate rather than overwrite.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports that the overwrite version silently gives the *wrong* gradient for the
tied-weight case (in one construction, the overwrite bug's `dL/dW` matched neither the correct
tied-weight gradient nor any obviously-related quantity at a glance — it's simply whichever of
$W$'s two backward contributions happened to run last in the topological order, with the earlier
one's contribution discarded rather than added), while the exact same buggy code gives the
*correct* gradient on the non-tied case, where every tensor has fan-out of exactly one (nothing
to overwrite over, since `_backward` for each node only ever runs once and each of its inputs
receives at most one contribution). A strong answer identifies the general principle: overwrite
vs. accumulate is invisible whenever the computation graph is a tree (every node has at most one
parent-consumer), and only matters once the graph has genuine fan-out — a value read by more than
one downstream operation, which is exactly the shape of the multivariate chain rule's *sum* over
all paths (the same rule this notebook's Theory section and `02.04`/`02.06` build on). Since
almost any nontrivial model reuses parameters or intermediate values in more than one place
(weight tying, residual/skip connections, any value used in more than one loss term), an
autodiff engine that overwrites instead of accumulates would be correct only by accident, on the
subset of graphs that happen to be trees — which is why every real engine, including
`torch.autograd`, accumulates gradients into `.grad` by design, and why `optimizer.zero_grad()`
(or the manual `p.grad = np.zeros_like(...)` this notebook's own training loop performs) exists
specifically to reset that accumulator between steps rather than relying on overwrite semantics.

</details>

---

## 📚 Further Reading
- Karpathy, [micrograd](https://github.com/karpathy/micrograd)
- Baydin et al., [\"Automatic Differentiation in Machine Learning: a Survey\"](https://arxiv.org/abs/1502.05767)
- Paszke et al., [\"PyTorch: An Imperative Style, High-Performance Deep Learning Library\"](https://arxiv.org/abs/1912.01703)

---

*Next notebook: [Capstone: Implement the Adam Optimizer](03_implement_adam_optimizer.ipynb)*